In [1]:
import os
import time
import requests
import pandas as pd
import numpy as np

In [2]:
DREMIO_URL = "http://localhost:9037"

DREMIO_USERNAME = "admin"      # sửa theo user Dremio của bạn
DREMIO_PASSWORD = "password123"   # sửa theo password Dremio của bạn

In [3]:
def dremio_login(base_url, username, password):
    login_url = f"{base_url}/apiv2/login"

    payload = {
        "userName": username,
        "password": password
    }

    response = requests.post(login_url, json=payload)

    if response.status_code != 200:
        raise Exception(
            f"Login Dremio failed: {response.status_code} - {response.text}"
        )

    token = response.json()["token"]
    return token


def run_dremio_sql(base_url, token, sql, poll_interval=2):
    headers = {
        "Authorization": f"_dremio{token}",
        "Content-Type": "application/json"
    }

    submit_url = f"{base_url}/api/v3/sql"

    response = requests.post(
        submit_url,
        headers=headers,
        json={"sql": sql}
    )

    if response.status_code not in [200, 202]:
        raise Exception(
            f"Submit SQL failed: {response.status_code} - {response.text}"
        )

    job_id = response.json()["id"]
    print(f"Dremio job_id: {job_id}")

    # Poll job status
    job_url = f"{base_url}/api/v3/job/{job_id}"

    while True:
        job_response = requests.get(job_url, headers=headers)

        if job_response.status_code != 200:
            raise Exception(
                f"Get job status failed: {job_response.status_code} - {job_response.text}"
            )

        job_info = job_response.json()
        job_state = job_info["jobState"]

        print(f"Job state: {job_state}")

        if job_state == "COMPLETED":
            break

        if job_state in ["FAILED", "CANCELED"]:
            raise Exception(f"Dremio job {job_state}: {job_info}")

        time.sleep(poll_interval)

    # Fetch results with pagination
    all_rows = []
    offset = 0
    limit = 500

    while True:
        result_url = f"{base_url}/api/v3/job/{job_id}/results?offset={offset}&limit={limit}"
        result_response = requests.get(result_url, headers=headers)

        if result_response.status_code != 200:
            raise Exception(
                f"Get results failed: {result_response.status_code} - {result_response.text}"
            )

        result_json = result_response.json()
        rows = result_json.get("rows", [])

        if not rows:
            break

        all_rows.extend(rows)

        if len(rows) < limit:
            break

        offset += limit

    df = pd.DataFrame(all_rows)
    return df

In [ ]:
token = dremio_login(
    DREMIO_URL,
    DREMIO_USERNAME,
    DREMIO_PASSWORD
)

sql = """
SELECT *
FROM nessie.gold.vw_ml_hotspot_dataset
WHERE target_hotspot_level IS NOT NULL
  AND target_next_listing_count IS NOT NULL
  AND listing_count IS NOT NULL
  AND snapshot_date IS NOT NULL
  AND area_name IS NOT NULL
  AND ward_name IS NOT NULL
  AND street_name IS NOT NULL
"""

df = run_dremio_sql(DREMIO_URL, token, sql)

print("Shape:", df.shape)
df.head()

Dremio job_id: 15c31674-9b69-378f-cd9d-9b447a39fa00
Job state: METADATA_RETRIEVAL
Job state: FAILED


Exception: Dremio job FAILED: {'jobState': 'FAILED', 'rowCount': 0, 'errorMessage': "Object 'vw_ml_hotspot_dataset' not found within 'nessie.gold'. Please check that it exists in the selected context.", 'startedAt': '2026-06-25T08:40:42.649Z', 'endedAt': '2026-06-25T08:40:42.696Z', 'queryType': 'REST', 'queueName': '', 'queueId': '', 'cancellationReason': ''}